# Compass Gate 1 — Qwen3-1.7B Hardware Check

This notebook measures whether Qwen3-1.7B can run on the selected Colab GPU. Use synthetic data only. Colab is development infrastructure; Compass should run locally for users.

In [ ]:
!nvidia-smi
!python --version

In [ ]:
%pip install -q -U transformers==4.57.1 accelerate==1.10.1 safetensors==0.6.2

In [ ]:
import json
import platform
import time
import torch
import transformers

MODEL_ID = 'Qwen/Qwen3-1.7B'
print(json.dumps({
    'python': platform.python_version(),
    'torch': torch.__version__,
    'transformers': transformers.__version__,
    'cuda_available': torch.cuda.is_available(),
    'cuda_version': torch.version.cuda,
    'model_id': MODEL_ID,
}, indent=2))

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

load_started = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map='auto',
)
load_seconds = time.perf_counter() - load_started
print(f'Loaded {MODEL_ID} in {load_seconds:.2f}s')
print(f'Device: {model.device}')
if torch.cuda.is_available():
    print(f'Peak allocated: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GiB')

In [ ]:
messages = [
    {'role': 'system', 'content': 'You are Compass. Do not calculate financial amounts. Route questions to approved deterministic tools.'},
    {'role': 'user', 'content': 'How much did I spend on groceries in May 2026?'},
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
started = time.perf_counter()
with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=96, do_sample=False)
elapsed = time.perf_counter() - started
new_tokens = output.shape[-1] - inputs['input_ids'].shape[-1]
response = tokenizer.decode(output[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)
print(response)
print(json.dumps({
    'latency_seconds': round(elapsed, 3),
    'new_tokens': int(new_tokens),
    'tokens_per_second': round(new_tokens / elapsed, 2) if elapsed else None,
    'peak_gpu_memory_gib': round(torch.cuda.max_memory_allocated() / 1024**3, 2) if torch.cuda.is_available() else None,
}, indent=2))

In [ ]:
# Save a compact evidence record. Do not include private or real financial data.
evidence = {
    'model_id': MODEL_ID,
    'license': 'Apache-2.0 (verify against the model card before release)',
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
    'gpu_memory_gib': round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2) if torch.cuda.is_available() else None,
    'python': platform.python_version(),
    'torch': torch.__version__,
    'transformers': transformers.__version__,
    'note': 'Synthetic smoke prompt only; this model did not calculate a financial amount.'
}
print(json.dumps(evidence, indent=2))
with open('qwen3_1_7b_hardware_evidence.json', 'w') as file:
    json.dump(evidence, file, indent=2)